# Schema Evolution в DuckLake

DuckLake поддерживает эволюцию схемы без downtime и без перезаписи данных:
- Добавление новых колонок (`ADD COLUMN`)
- Переименование колонок (`RENAME COLUMN`)
- Изменение типов (`ALTER COLUMN TYPE`)

Старые Parquet-файлы остаются неизменными. DuckLake подставляет
NULL для новых колонок при чтении старых файлов — прозрачно для запросов.

In [ ]:
import os, sys
sys.path.insert(0, '..')

os.environ.setdefault('DUCKLAKE_PG_HOST', 'localhost')
os.environ.setdefault('DUCKLAKE_PG_DB', 'ducklake_catalog')
os.environ.setdefault('DUCKLAKE_PG_USER', 'ducklake')
os.environ.setdefault('DUCKLAKE_PG_PASSWORD', 'ducklake_secret_change_me')
os.environ.setdefault('MINIO_ACCESS_KEY', 'minioadmin')
os.environ.setdefault('MINIO_SECRET_KEY', 'minioadmin')
os.environ.setdefault('MINIO_ENDPOINT', 'http://localhost:9000')
os.environ.setdefault('MINIO_BUCKET', 'ducklake-flights')

from src.generators.connection import get_ducklake_connection
conn = get_ducklake_connection()
print('Connected!')

## Текущая схема таблицы bookings

In [ ]:
import pandas as pd

schema_before = conn.execute("DESCRIBE flights.bookings").df()
print('Схема до изменений:')
display(schema_before)

snap_before = conn.execute(
    "SELECT MAX(snapshot_id) FROM ducklake_snapshots('flights')"
).fetchone()[0]
print(f'Текущий снэпшот: #{snap_before}')

## Добавление новой колонки

In [ ]:
# Добавляем колонку loyalty_points — без перезаписи данных!
conn.execute("""
    ALTER TABLE flights.bookings
    ADD COLUMN loyalty_points INTEGER DEFAULT 0
""")
print('Колонка loyalty_points добавлена.')

# Заполняем для новых записей
conn.execute("""
    UPDATE flights.bookings
    SET loyalty_points = CAST(price_rub / 100 AS INTEGER)
    WHERE loyalty_points = 0
      AND status = 'boarded'
""")

snap_after_add = conn.execute(
    "SELECT MAX(snapshot_id) FROM ducklake_snapshots('flights')"
).fetchone()[0]
print(f'Снэпшот после ADD COLUMN: #{snap_after_add}')

## Проверка: старые данные читаются с NULL в новой колонке

In [ ]:
# Текущие данные — loyalty_points заполнен
df_now = conn.execute("""
    SELECT booking_id, fare_class, price_rub, loyalty_points, status
    FROM flights.bookings
    WHERE status = 'boarded'
    LIMIT 5
""").df()
print('Текущие данные (loyalty_points заполнен):')
display(df_now)

# Данные из старого снэпшота — колонки ещё не было
df_old = conn.execute(f"""
    SELECT booking_id, fare_class, price_rub, loyalty_points, status
    FROM flights.bookings AT (VERSION => {snap_before})
    WHERE status = 'boarded'
    LIMIT 5
""").df()
print(f'Данные из снэпшота #{snap_before} (до ADD COLUMN — loyalty_points = NULL):')
display(df_old)

## Переименование колонки

In [ ]:
conn.execute("""
    ALTER TABLE flights.bookings
    RENAME COLUMN loyalty_points TO miles_earned
""")
print('Колонка переименована: loyalty_points → miles_earned')

schema_after = conn.execute("DESCRIBE flights.bookings").df()
print('Схема после изменений:')
display(schema_after)

## Откат схемы (cleanup)

In [ ]:
# Убираем демонстрационную колонку чтобы не ломать dbt-модели
conn.execute("ALTER TABLE flights.bookings DROP COLUMN miles_earned")
print('Колонка miles_earned удалена. Схема восстановлена.')

schema_restored = conn.execute("DESCRIBE flights.bookings").df()
display(schema_restored)

In [ ]:
conn.close()
print('Done.')